In [68]:
import pandas as pd

In [69]:
df=pd.read_csv("/Users/vijaypatidar/vijay/Anshul/deliver time prediction/data/Food_Delivery_Data.csv")
df.sample(5)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
25654,0xb3af,RANCHIRES15DEL01,33.0,4.9,23.369746,85.339820,23.379746,85.349820,05-04-2022,9:41:09,9:59:43,conditions Fog,Low,1,Buffet,motorcycle,1.0,No,Metropolitian,(min) 15
35932,0x8fa9,BANGRES18DEL01,22.0,4.7,12.913041,77.683237,12.923041,77.693237,26-03-2022,8:58:10,9:00:34,conditions Windy,Low,2,Drinks,electric_scooter,0.0,No,Metropolitian,(min) 10
21406,0x1774,BANGRES19DEL01,36.0,4.8,12.914264,77.678400,13.024264,77.788400,10-03-2022,22:49:29,22:54:38,conditions Windy,Low,2,Meal,motorcycle,1.0,No,Urban,(min) 20
36702,0x1bba,BANGRES16DEL03,20.0,4.8,13.029198,77.570997,13.059198,77.600997,26-03-2022,20:22:08,20:27:28,conditions Sandstorms,Jam,1,Snack,scooter,1.0,No,Urban,(min) 18
31733,0xae47,HYDRES05DEL03,25.0,4.8,17.433809,78.386744,17.493809,78.446744,09-03-2022,22:23:56,22:39:26,conditions Sunny,Low,2,Snack,scooter,0.0,No,Urban,(min) 16


In [70]:
from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in KM

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c

In [71]:
df["Delivery_Distance"] = df.apply(
    lambda row: haversine_distance(
        row["Restaurant_latitude"],
        row["Restaurant_longitude"],
        row["Delivery_location_latitude"],
        row["Delivery_location_longitude"]
    ),
    axis=1
)

In [72]:
df["Delivery_Distance"].head()

0     3.025149
1    20.183530
2     1.552758
3     7.790401
4     6.210138
Name: Delivery_Distance, dtype: float64

In [73]:
df["Delivery_Distance"].describe()

count    41953.000000
mean         9.720284
std          5.603083
min          1.465067
25%          4.657655
50%          9.193021
75%         13.680920
max         20.969489
Name: Delivery_Distance, dtype: float64

In [74]:
df[
    [
        "Time_Orderd",
        "Time_Order_picked"
    ]
].dtypes

Time_Orderd          object
Time_Order_picked    object
dtype: object

In [75]:
df["Time_Orderd"] = pd.to_datetime(
    df["Time_Orderd"],
    format="%H:%M:%S",
    errors="coerce"
)

df["Time_Order_picked"] = pd.to_datetime(
    df["Time_Order_picked"],
    format="%H:%M:%S",
    errors="coerce"
)

In [76]:
df["Preparation_Time"] = (
    df["Time_Order_picked"] -
    df["Time_Orderd"]
).dt.total_seconds() / 60

df.loc[
    df["Preparation_Time"] < 0,
    "Preparation_Time"
] += 24 * 60

In [77]:
df[
    [
        "Time_Orderd",
        "Time_Order_picked",
        "Preparation_Time"
    ]
].head()

,Time_Orderd,Time_Order_picked,Preparation_Time
0,1900-01-01 11:33:33,1900-01-01 11:45:29,11.933333
1,1900-01-01 19:45:37,1900-01-01 19:51:49,6.200000
2,1900-01-01 08:32:58,1900-01-01 08:48:47,15.816667
3,1900-01-01 18:03:58,1900-01-01 18:12:52,8.900000
4,1900-01-01 13:34:16,1900-01-01 13:45:36,11.333333


In [78]:
df["Preparation_Time"].describe()

count    40353.000000
mean        12.296423
std         63.384520
min          0.000000
25%          5.783333
50%          9.500000
75%         13.266667
max       1439.983333
Name: Preparation_Time, dtype: float64

In [79]:
# Production Feature Set
X_production = df.drop(
    columns=[
        "Preparation_Time",
        "Time_taken(min)"
    ]
)

# Experimental Feature Set
X_experiment = df.drop(
    columns=[
        "Time_taken(min)"
    ]
)

In [80]:
drop_columns = [
    "ID",
    "Delivery_person_ID",

    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude",

    "Order_Date",
    "Time_Orderd",
    "Time_Order_picked",

    "Preparation_Time"
]

In [83]:
df_model=df.drop(columns=drop_columns)

In [86]:
df_model.sample(5)

,Delivery_person_Age,Delivery_person_Ratings,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min),Delivery_Distance
2461,NaN,NaN,conditions NaN,NaN,0,Meal,motorcycle,0.0,No,Metropolitian,(min) 28,9.316114
30627,32.0,5.0,conditions Cloudy,Medium,1,Meal,scooter,0.0,No,Metropolitian,(min) 33,13.611585
12358,NaN,NaN,conditions Cloudy,Medium,1,Buffet,motorcycle,1.0,No,NaN,(min) 30,13.481326
14090,NaN,NaN,conditions Windy,Jam,2,Buffet,electric_scooter,1.0,No,Metropolitian,(min) 29,12.236724
26042,31.0,4.4,conditions Fog,Jam,2,Drinks,motorcycle,1.0,No,Metropolitian,(min) 38,10.868244


In [87]:
df_model.shape

(41953, 12)

In [88]:
df_model.columns.tolist()

['Delivery_person_Age',
 'Delivery_person_Ratings',
 'Weatherconditions',
 'Road_traffic_density',
 'Vehicle_condition',
 'Type_of_order',
 'Type_of_vehicle',
 'multiple_deliveries',
 'Festival',
 'City',
 'Time_taken(min)',
 'Delivery_Distance']

In [91]:
df_model.to_csv(
    "../data/food_delivery_processed.csv",
    index=False
)